# Practical 4: Tic-Tac-Toe Game-Playing Agent

**Objective:** Develop a game-playing agent using Minimax and Alpha-Beta Pruning for Tic-Tac-Toe.

This practical focuses on **one small game-playing agent only**.

We will build a Tic-Tac-Toe AI where:

- `X` is the AI.
- `O` is the human player.
- **Minimax** helps the AI choose the best move.
- **Alpha-Beta Pruning** lets Minimax skip moves that do not need to be checked.

The code is kept small, but every part is explained carefully.

## 1. The Tic-Tac-Toe Board

We use a Python list with **9 positions**.

The position numbers are:

```text
0 | 1 | 2
--+---+--
3 | 4 | 5
--+---+--
6 | 7 | 8
```

Each position contains:

- `'X'` for the AI,
- `'O'` for the player,
- `' '` for an empty position.

In [1]:
board = ['']*9
print(board)

['', '', '', '', '', '', '', '', '']


### Understanding the code

```python
[' '] * 9
```

creates a list containing 9 empty spaces.

This represents an empty Tic-Tac-Toe board.

## 2. Display the Board

The list is useful for Python, but it is easier for us to understand the game if we display it like a real Tic-Tac-Toe board.

In [2]:
def show_board(board):
    print()
    print(board[0], '|', board[1], '|', board[2])
    print('-+--+-')
    print(board[3], '|', board[4], '|', board[5])
    print('-+--+-')
    print(board[6], '|', board[7], '|', board[8])
    print('-+--+-')
show_board(board)


 |  | 
-+--+-
 |  | 
-+--+-
 |  | 
-+--+-


## 3. Check for a Winner

A player wins when three matching marks appear in one line.

There are 8 possible winning lines:

- 3 rows,
- 3 columns,
- 2 diagonals.

In [3]:
WINNING_LINES = [
    [0, 1, 2], [3, 4, 5], [6, 7, 8],
    [0, 3, 6], [1, 4, 7], [2, 5, 8],
    [0, 4, 8], [2, 4, 6]
]


def winner(board):
    for a,b,c in WINNING_LINES:
        if board[a] != '' and board[a]==board[b]==board[c]:
            return board[a]
    return None

def empty_boxes(board):
    return [i for i in range(9) if board[i] == ' ']

### Understanding these helper functions

`winner(board)` checks each possible winning line.

If three positions contain the same mark, it returns either `'X'` or `'O'`.

If nobody has won, it returns `None`.

`empty_boxes(board)` returns the position numbers that are still empty.

For example, if only positions 2 and 5 are empty, it returns:

```python
[2, 5]
```

## 4. The Main Idea of Minimax

Minimax looks ahead at possible future moves.

We give the final result of a game a score:

- AI (`X`) wins → `+1`
- Player (`O`) wins → `-1`
- Draw → `0`

The AI is the **MAX player** because it wants the highest score.

The human player is the **MIN player** because Minimax assumes the opponent will choose moves that give the AI the lowest score.

> **Remember:** MAX chooses the biggest score. MIN chooses the smallest score.

## 5. What Is Alpha-Beta Pruning?

Normal Minimax may check many possible moves.

Alpha-Beta Pruning lets it stop checking a branch when that branch cannot change the final choice.

We keep two values:

- `alpha` = the best score MAX already knows it can get.
- `beta` = the best score MIN already knows it can get.

When:

```text
beta <= alpha
```

we can stop checking the rest of that branch.

This is called **pruning**.

> **Important:** Alpha-Beta Pruning does not change the answer. It only avoids checking some unnecessary branches.

## 6. Write Minimax with Alpha-Beta Pruning

We will now put both ideas into one function.

In [4]:
def minimax(board, is_max_turn, alpha, beta):
    # Check whether the game is already finished
    win = winner(board)

    if win == 'X':
        return 1

    if win == 'O':
        return -1

    if len(empty_boxes(board)) == 0:
        return 0  

    # MAX turn: AI chooses the highest score
    if is_max_turn:
        best_score = -2

        for box in empty_boxes(board):
            board[box] = 'X'
            score = minimax(board, False, alpha, beta)
            board[box] = ' '

            best_score = max(best_score, score)
            alpha = max(alpha, best_score)

            if beta <= alpha:
                break

        return best_score

    # MIN turn: opponent chooses the lowest score
    else:
        best_score = 2

        for box in empty_boxes(board):
            board[box] = 'O'
            score = minimax(board, True, alpha, beta)
            board[box] = ' '

            best_score = min(best_score, score)
            beta = min(beta, best_score)

            if beta <= alpha:
                break

        return best_score

## 7. Understanding the Minimax Code

### Part 1: Stop at the end of the game

```python
if win == 'X':
    return 1
if win == 'O':
    return -1
if len(empty_boxes(board)) == 0:
    return 0
```

These are the final scores.

The recursive search stops when someone wins or there are no empty positions left.

### Part 2: MAX tries every possible `X` move

```python
for box in empty_boxes(board):
    board[box] = 'X'
    score = minimax(board, False, alpha, beta)
    board[box] = ' '
```

The AI temporarily places `X` in an empty position.

Then Minimax checks what could happen after that move.

After checking, the position is changed back to empty.

This lets the program try another possible move.

### Part 3: MAX keeps the highest score

```python
best_score = max(best_score, score)
alpha = max(alpha, best_score)
```

`best_score` remembers the best result found for the AI.

`alpha` remembers the best score MAX already knows it can get.

### Part 4: MIN tries the opponent's moves

When it is the opponent's turn, the same idea is used with `O`.

But MIN keeps the **smallest** score:

```python
best_score = min(best_score, score)
beta = min(beta, best_score)
```

`beta` remembers the best score MIN already knows it can get.

### Part 5: Alpha-Beta Pruning

Both parts contain:

```python
if beta <= alpha:
    break
```

This means the remaining moves in that branch do not need to be checked.

The program stops exploring that branch and continues somewhere else.

## 8. Let the AI Choose Its Best Move

The Minimax function gives us the score of a possible move.

Now we need a function that tries every available move and returns the best position for `X`.

In [5]:
def best_move(board):
    best_score = -2
    best_box = None

    for box in empty_boxes(board):
        board[box] = 'X'
        score = minimax(board, False, -2, 2)
        board[box] = ' '

        if score > best_score:
            best_score = score
            best_box = box

    return best_box

### Why do we start with `-2` and `2`?

Our possible game scores are only:

`-1`, `0`, and `1`.

So `-2` is safely below every possible score, and `2` is safely above every possible score.

This gives Alpha-Beta Pruning its starting limits.

## 9. Test the Agent

We will give the AI a board where it can win immediately.

The board is:

```text
X | O | X
--+---+--
O | X |  
--+---+--
  |   | O
```

Position `6` gives `X` a diagonal win.

In [6]:
test_board = [
    'X', 'O', 'X',
    'O', 'X', ' ',
    ' ', ' ', 'O'
]

show_board(test_board)

move = best_move(test_board)

print('AI chooses position:', move)


X | O | X
-+--+-
O | X |  
-+--+-
  |   | O
-+--+-
AI chooses position: 6


### Expected Result

The AI should choose:

```text
6
```

because placing `X` at position `6` completes the diagonal `2 → 4 → 6`.

## 10. Play Against the Agent

The final function lets a human play against the AI.

The AI uses `X` and moves first.

The human uses `O` and enters a position from `0` to `8`.

In [7]:
def play_game():
    board = [' '] * 9

    print('You are O. The AI is X.')
    print('Choose positions using numbers 0 to 8.')

    while True:
        # AI move
        ai_move = best_move(board)
        board[ai_move] = 'X'

        print('\nAI chooses:', ai_move)
        show_board(board)

        if winner(board) == 'X':
            print('AI wins!')
            break

        if len(empty_boxes(board)) == 0:
            print('Draw!')
            break

        # Human move
        while True:
            try:
                player_move = int(input('Your move (0-8): '))

                if player_move in empty_boxes(board):
                    break

                print('That position is not available.')

            except ValueError:
                print('Please enter a number from 0 to 8.')

        board[player_move] = 'O'
        show_board(board)

        if winner(board) == 'O':
            print('You win!')
            break

        if len(empty_boxes(board)) == 0:
            print('Draw!')
            break
play_game()

### Start the game

Run the next cell when you are ready to play.

In [8]:
play_game()

You are O. The AI is X.
Choose positions using numbers 0 to 8.

AI chooses: 0

X |   |  
-+--+-
  |   |  
-+--+-
  |   |  
-+--+-
Your move (0-8): 2

X |   | O
-+--+-
  |   |  
-+--+-
  |   |  
-+--+-

AI chooses: 3

X |   | O
-+--+-
X |   |  
-+--+-
  |   |  
-+--+-
Your move (0-8): 6

X |   | O
-+--+-
X |   |  
-+--+-
O |   |  
-+--+-

AI chooses: 4

X |   | O
-+--+-
X | X |  
-+--+-
O |   |  
-+--+-
Your move (0-8): 8

X |   | O
-+--+-
X | X |  
-+--+-
O |   | O
-+--+-

AI chooses: 5

X |   | O
-+--+-
X | X | X
-+--+-
O |   | O
-+--+-
AI wins!


## 11. Quick Recap

In this practical, we developed a Tic-Tac-Toe game-playing agent.

The important ideas are:

1. The board is stored as a list of 9 positions.
2. Minimax checks possible future game results.
3. `X` is MAX and tries to get the highest score.
4. `O` is MIN and is assumed to choose the lowest score for `X`.
5. Alpha-Beta Pruning skips branches that cannot change the final decision.
6. `best_move()` uses Minimax with Alpha-Beta Pruning to choose the AI's move.

> **Main idea:** Minimax chooses the best move, while Alpha-Beta Pruning helps it reach the same decision with less searching.